# Train YOLO nano cho mục tiêu mô phỏng (G4 phần B)

Chạy trên Google Colab / Kaggle (GPU T4). Dữ liệu sinh bằng `python perception/make_dataset.py --n 4000` rồi nén thư mục `perception/data/sim_person` thành `sim_person.zip` và tải lên.

Đầu ra: `yolo_target.onnx` (imgsz 320) + `yolo_eval.json` (mAP50 trên tập **test** mô phỏng, chia theo seed cảnh). Chép cả hai về `perception/weights/` và `docs/` trên laptop.

In [ ]:
!pip -q install ultralytics onnx onnxruntime
import ultralytics, torch
print(ultralytics.__version__, torch.__version__, torch.cuda.is_available())

In [ ]:
# Tải dataset lên (zip của perception/data/sim_person) — hoặc mount Drive
from google.colab import files  # noqa
up = files.upload()
import zipfile, pathlib
zip_name = next(iter(up))
zipfile.ZipFile(zip_name).extractall('/content/data')
root = next(p for p in pathlib.Path('/content/data').rglob('data.yaml')).parent
yaml_txt = f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: person\n"
(root / 'data.yaml').write_text(yaml_txt)
print(yaml_txt)

In [ ]:
from ultralytics import YOLO
# bản nano mới nhất mà Ultralytics cung cấp (đổi tên nếu bản mới hơn tồn tại, ví dụ yolo26n.pt)
MODEL = 'yolo11n.pt'
model = YOLO(MODEL)
res = model.train(data=str(root / 'data.yaml'), imgsz=320, epochs=60, batch=64, seed=0, deterministic=True,
                  project='/content/runs', name='sim_person', exist_ok=True, patience=15,
                  mosaic=1.0, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, degrees=180, flipud=0.5, fliplr=0.5)

In [ ]:
# mAP50 trên tập TEST (chưa dùng khi train/val)
import json
best = YOLO('/content/runs/sim_person/weights/best.pt')
m = best.val(data=str(root / 'data.yaml'), split='test', imgsz=320, batch=64, plots=False)
ev = {'model': MODEL, 'imgsz': 320, 'split': 'test', 'map50': float(m.box.map50), 'map50_95': float(m.box.map),
      'precision': float(m.box.mp), 'recall': float(m.box.mr), 'n_test_images': int(len(list((root/'images'/'test').glob('*.jpg')))),
      'ultralytics': ultralytics.__version__, 'epochs': 60}
json.dump(ev, open('/content/yolo_eval.json', 'w'), indent=2)
print(json.dumps(ev, indent=2))

In [ ]:
# Xuất ONNX (opset 12, imgsz 320, không dynamic để onnxruntime CPU chạy nhanh)
onnx_path = best.export(format='onnx', imgsz=320, opset=12, simplify=True, dynamic=False)
import shutil
shutil.copy(onnx_path, '/content/yolo_target.onnx')
files.download('/content/yolo_target.onnx')
files.download('/content/yolo_eval.json')

Trên laptop:
```
copy yolo_target.onnx perception/weights/
copy yolo_eval.json docs/
python scripts/bench_yolo.py            # FPS CPU → docs/yolo_bench.json
python evaluate.py --run results/<g4_run> --controller ppo --source yolo --episodes 100
```